In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "Fraud-detection-ML-V2" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
FIGURES_DIR = OUTPUTS_DIR / "figures"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FEATURE_PATH = PROCESSED_DATA_DIR / "ml_training_features.csv"

if not TRAIN_FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Training feature file not found:\n{TRAIN_FEATURE_PATH}"
    )

df = pd.read_csv(TRAIN_FEATURE_PATH)

ML_FEATURES = [
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]

TARGET_COLUMN = "is_fraud"

required_columns = ML_FEATURES + [TARGET_COLUMN]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("========== PHASE 4 DATA LOADED ==========")
print("Dataset shape:", df.shape)
print("Number of ML features:", len(ML_FEATURES))
print("ML features:", ML_FEATURES)
print("Target:", TARGET_COLUMN)
print("=========================================")

========== PHASE 4 DATA LOADED ==========
Dataset shape: (1296675, 9)
Number of ML features: 6
ML features: ['amount', 'amount_vs_avg_ratio', 'txn_count_last_5min', 'time_since_last_txn_sec', 'distance_from_last_location_km', 'merchant_category_is_new_for_user']
Target: is_fraud


In [2]:
for column in ML_FEATURES:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df[TARGET_COLUMN] = pd.to_numeric(
    df[TARGET_COLUMN],
    errors="coerce"
)

if df[ML_FEATURES].isnull().any().any():
    raise ValueError(
        "Missing values detected in ML features."
    )

if np.isinf(
    df[ML_FEATURES].to_numpy(
        dtype=float
    )
).any():
    raise ValueError(
        "Infinite values detected in ML features."
    )

if not df[TARGET_COLUMN].isin([0, 1]).all():
    raise ValueError(
        "Target contains values other than 0 and 1."
    )

print("========== FEATURE VALIDATION ==========")
print(
    "Missing feature values:",
    int(
        df[ML_FEATURES]
        .isnull()
        .sum()
        .sum()
    )
)

print(
    "Infinite feature values:",
    int(
        np.isinf(
            df[ML_FEATURES].to_numpy(
                dtype=float
            )
        ).sum()
    )
)

print(
    "Target values:",
    sorted(
        df[TARGET_COLUMN].unique().tolist()
    )
)

print(
    "Feature validation passed:",
    True
)

print("=========================================")

========== FEATURE VALIDATION ==========
Missing feature values: 0
Infinite feature values: 0
Target values: [0, 1]
Feature validation passed: True


In [3]:
X = df[ML_FEATURES].copy()
y = df[TARGET_COLUMN].astype(int).copy()

print("========== MODEL INPUTS ==========")
print("X shape:", X.shape)
print("y shape:", y.shape)
print()
print("X columns:")
print(X.columns.tolist())
print()
print("Target distribution:")
print(y.value_counts().sort_index())
print("==================================")

========== MODEL INPUTS ==========
X shape: (1296675, 6)
y shape: (1296675,)

X columns:
['amount', 'amount_vs_avg_ratio', 'txn_count_last_5min', 'time_since_last_txn_sec', 'distance_from_last_location_km', 'merchant_category_is_new_for_user']

Target distribution:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64


In [4]:
negative_count = int(
    (y == 0).sum()
)

positive_count = int(
    (y == 1).sum()
)

if positive_count == 0:
    raise ValueError(
        "No fraud transactions found in the training data."
    )

scale_pos_weight = (
    negative_count / positive_count
)

print("========== CLASS IMBALANCE ==========")
print("Legitimate transactions:", negative_count)
print("Fraud transactions:", positive_count)
print(
    "Fraud percentage:",
    round(
        positive_count / len(y) * 100,
        6
    )
)
print(
    "scale_pos_weight:",
    scale_pos_weight
)
print("=====================================")

========== CLASS IMBALANCE ==========
Legitimate transactions: 1289169
Fraud transactions: 7506
Fraud percentage: 0.578865
scale_pos_weight: 171.75179856115108


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("========== TRAIN / VALIDATION SPLIT ==========")
print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)
print()
print("Training target distribution:")
print(y_train.value_counts().sort_index())
print()
print("Validation target distribution:")
print(y_valid.value_counts().sort_index())
print("==============================================")

========== TRAIN / VALIDATION SPLIT ==========
Training features: (1037340, 6)
Validation features: (259335, 6)

Training target distribution:
is_fraud
0    1031335
1       6005
Name: count, dtype: int64

Validation target distribution:
is_fraud
0    257834
1      1501
Name: count, dtype: int64


In [6]:
from xgboost import XGBClassifier

baseline_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),
        (X_valid, y_valid)
    ],
    verbose=False
)

print("========== BASELINE XGBOOST ==========")
print("Model trained successfully:", True)
print("Number of features:", len(ML_FEATURES))
print("Number of estimators:", 300)
print("======================================")

========== BASELINE XGBOOST ==========
Model trained successfully: True
Number of features: 6
Number of estimators: 300


In [7]:
y_valid_probability = baseline_model.predict_proba(
    X_valid
)[:, 1]

y_valid_prediction = (
    y_valid_probability >= 0.5
).astype(int)

print("========== VALIDATION PREDICTIONS ==========")
print(
    "Probability count:",
    len(y_valid_probability)
)

print(
    "Prediction count:",
    len(y_valid_prediction)
)

print(
    "Minimum probability:",
    float(y_valid_probability.min())
)

print(
    "Maximum probability:",
    float(y_valid_probability.max())
)

print("============================================")

========== VALIDATION PREDICTIONS ==========
Probability count: 259335
Prediction count: 259335
Minimum probability: 0.0001172607735497877
Maximum probability: 0.9992818236351013


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(
    y_valid,
    y_valid_prediction
)

precision = precision_score(
    y_valid,
    y_valid_prediction,
    zero_division=0
)

recall = recall_score(
    y_valid,
    y_valid_prediction,
    zero_division=0
)

f1 = f1_score(
    y_valid,
    y_valid_prediction,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_valid,
    y_valid_probability
)

pr_auc = average_precision_score(
    y_valid,
    y_valid_probability
)

confusion = confusion_matrix(
    y_valid,
    y_valid_prediction
)

print("========== BASELINE METRICS ==========")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)
print()
print("Confusion Matrix:")
print(confusion)
print()
print("Classification Report:")
print(
    classification_report(
        y_valid,
        y_valid_prediction,
        zero_division=0
    )
)
print("======================================")

========== BASELINE METRICS ==========
Accuracy: 0.9339772880637014
Precision: 0.07214461791290057
Recall: 0.8774150566289141
F1 Score: 0.1333265843288115
ROC-AUC: 0.9723982760558897
PR-AUC: 0.44958326241188656

Confusion Matrix:
[[240896  16938]
 [   184   1317]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.93      0.97    257834
           1       0.07      0.88      0.13      1501

    accuracy                           0.93    259335
   macro avg       0.54      0.91      0.55    259335
weighted avg       0.99      0.93      0.96    259335



In [9]:
baseline_metrics = {
    "model": "XGBoost",
    "model_type": "baseline",
    "feature_count": len(ML_FEATURES),
    "features": ML_FEATURES,
    "scale_pos_weight": float(scale_pos_weight),
    "threshold": 0.5,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "training_rows": int(len(X_train)),
    "validation_rows": int(len(X_valid))
}

BASELINE_METRICS_PATH = (
    METRICS_DIR
    / "xgboost_baseline_metrics.json"
)

with open(
    BASELINE_METRICS_PATH,
    "w"
) as file:
    json.dump(
        baseline_metrics,
        file,
        indent=4
    )

print("========== BASELINE METRICS SAVED ==========")
print("File:", BASELINE_METRICS_PATH)
print(
    "File exists:",
    BASELINE_METRICS_PATH.exists()
)
print("=============================================")

========== BASELINE METRICS SAVED ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\xgboost_baseline_metrics.json
File exists: True


In [10]:
BASELINE_MODEL_PATH = (
    MODELS_DIR
    / "xgboost_baseline.json"
)

baseline_model.save_model(
    BASELINE_MODEL_PATH
)

print("========== BASELINE MODEL SAVED ==========")
print("Model path:", BASELINE_MODEL_PATH)
print(
    "Model exists:",
    BASELINE_MODEL_PATH.exists()
)
print("==========================================")

========== BASELINE MODEL SAVED ==========
Model path: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\xgboost_baseline.json
Model exists: True


In [11]:
FEATURE_COLUMNS_PATH = (
    MODELS_DIR
    / "feature_columns.json"
)

feature_contract = {
    "features": ML_FEATURES,
    "feature_count": len(ML_FEATURES)
}

with open(
    FEATURE_COLUMNS_PATH,
    "w"
) as file:
    json.dump(
        feature_contract,
        file,
        indent=4
    )

print("========== FEATURE CONTRACT ==========")
print(
    "Feature file:",
    FEATURE_COLUMNS_PATH
)

print(
    "Feature file exists:",
    FEATURE_COLUMNS_PATH.exists()
)

print(
    "Feature count:",
    len(ML_FEATURES)
)

print("======================================")

========== FEATURE CONTRACT ==========
Feature file: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\feature_columns.json
Feature file exists: True
Feature count: 6


In [12]:
from xgboost import XGBClassifier

reloaded_model = XGBClassifier()

reloaded_model.load_model(
    BASELINE_MODEL_PATH
)

reloaded_probability = reloaded_model.predict_proba(
    X_valid.head(10)
)[:, 1]

if not np.isfinite(
    reloaded_probability
).all():
    raise ValueError(
        "Reloaded model produced invalid probabilities."
    )

print("========== MODEL RELOAD TEST ==========")
print(
    "Model reloaded successfully:",
    True
)

print(
    "Test predictions generated:",
    len(reloaded_probability)
)

print(
    "Probabilities valid:",
    True
)

print("=======================================")

========== MODEL RELOAD TEST ==========
Model reloaded successfully: True
Test predictions generated: 10
Probabilities valid: True


In [13]:
print()
print("================================================")
print("     STREAMSENTINEL V2 — PHASE 4 SUMMARY")
print("================================================")

print()

print(
    "Model:",
    "XGBoost"
)

print(
    "Number of ML features:",
    len(ML_FEATURES)
)

print(
    "Features:",
    ML_FEATURES
)

print(
    "Training rows:",
    len(X_train)
)

print(
    "Validation rows:",
    len(X_valid)
)

print(
    "Fraud training rows:",
    int((y_train == 1).sum())
)

print(
    "Legitimate training rows:",
    int((y_train == 0).sum())
)

print()

print("Baseline Metrics")
print("----------------")
print(
    "Accuracy:",
    accuracy
)

print(
    "Precision:",
    precision
)

print(
    "Recall:",
    recall
)

print(
    "F1:",
    f1
)

print(
    "ROC-AUC:",
    roc_auc
)

print(
    "PR-AUC:",
    pr_auc
)

print()

print(
    "Baseline model saved:",
    BASELINE_MODEL_PATH.exists()
)

print(
    "Feature contract saved:",
    FEATURE_COLUMNS_PATH.exists()
)

print(
    "Baseline metrics saved:",
    BASELINE_METRICS_PATH.exists()
)

print()
print("PHASE 4 XGBOOST MODEL DEVELOPMENT: COMPLETE")
print("================================================")


     STREAMSENTINEL V2 — PHASE 4 SUMMARY

Model: XGBoost
Number of ML features: 6
Features: ['amount', 'amount_vs_avg_ratio', 'txn_count_last_5min', 'time_since_last_txn_sec', 'distance_from_last_location_km', 'merchant_category_is_new_for_user']
Training rows: 1037340
Validation rows: 259335
Fraud training rows: 6005
Legitimate training rows: 1031335

Baseline Metrics
----------------
Accuracy: 0.9339772880637014
Precision: 0.07214461791290057
Recall: 0.8774150566289141
F1: 0.1333265843288115
ROC-AUC: 0.9723982760558897
PR-AUC: 0.44958326241188656

Baseline model saved: True
Feature contract saved: True
Baseline metrics saved: True

PHASE 4 XGBOOST MODEL DEVELOPMENT: COMPLETE
